# Notebook 5 — IFC2SB: controle e creche integral · v01

**Objetivo:** verificar se uma implementação publicada consegue produzir limites de ambientes utilizáveis como entrada para o enriquecimento semântico. O motor é o **IFC2SB dos autores**, em C++, associado a Fichter et al. (2021). Os operadores geométricos não são substituídos por uma implementação Python.

Este notebook reproduz a **auditoria das execuções efetivamente realizadas no Colab**, incluindo os registros integrais. A recompilação e uma nova execução C++ são etapas separadas, descritas ao final. As células Python consultam entidades, verificam arquivos e organizam evidências; são rotinas específicas deste estudo, fundamentadas na API oficial.

Não se declara a ontologia arquitetônica completa nem se resolve a integração com a estrutura. O IFC FNDE de origem permanece preservado.


## Fontes, código e diferenças de ambiente

- FICHTER, Eric; RICHTER, Veronika; FRISCH, Jérôme; VAN TREECK, Christoph. **Automatic generation of second level space boundary geometry from IFC models**. Building Simulation 2021, p. 1083–1090. [DOI](https://doi.org/10.26868/25222708.2021.30156) · [PDF](https://github.com/prcgui/bim-ontology-CIV2155/blob/main/referencias/geometria_relacoes/Fichter_et_al_2021_Geracao_Limites_Espaciais.pdf).
- [Código dos autores, commit 96a89cdf](https://github.com/RWTH-E3D/IFC2SB/tree/96a89cdff65f13279c0c3ce67e729ad9f9f5adad): `src/Graph.cpp`, `src/main.cpp`, `add/Dockerfile`; licença GPL-3.0 conforme LICENSE. O filtro original exclui IfcCovering. Não se aplicou a alteração experimental desse filtro nestes dois ensaios.
- [Modelo de controle no repositório institucional IBPSA](https://github.com/ibpsa/project1-wp-2-2-bim/blob/a3cba6418feb17f7381b435cccc1f02432e35782/IFC_Files/MISC/AC20-FZK-Haus.ifc), crédito IAI/KIT. Tem o mesmo nome do exemplo do README do IFC2SB; não se afirma identidade binária com o exemplar dos autores.
- [IfcOpenShell: exemplos oficiais](https://docs.ifcopenshell.org/ifcopenshell-python/code_examples.html): leitura, `by_type`, atributos e inversas. Auditoria realizada com **0.8.4**, distinta da biblioteca C++ 0.6.0 usada na compilação do motor.
- [buildingSMART IFC4: IfcRelSpaceBoundary2ndLevel](https://standards.buildingsmart.org/IFC/RELEASE/IFC4/ADD2_TC1/HTML/link/ifcrelspaceboundary2ndlevel.htm): interpretação das entidades de limite.
- Ajustes de compatibilidade estão preservados em `ambiente/compatibilidade*.diff`; fontes técnicas OCCT constam dos arquivos JSON correspondentes. O ambiente OCCT 7.6.3 difere do ambiente originalmente proposto pelos autores. Trata-se de execução do código publicado com adaptações de compilação registradas, não reprodução exata de benchmark.


## 1. Dependência e registros verificáveis

Baixa os registros publicados somente quando ausentes e confere seus hashes. O pacote contém a saída original do controle, os logs e os manifestos. Não confundir o download desses resultados com nova execução do motor.


In [1]:
from pathlib import Path
import os,sys,json,hashlib,zipfile,subprocess,importlib.util
from urllib.request import urlretrieve
if importlib.util.find_spec('ifcopenshell') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','ifcopenshell==0.8.4'])
import ifcopenshell
print('IfcOpenShell da auditoria:',ifcopenshell.version)
assert ifcopenshell.version=='0.8.4','Usar a versão declarada antes de comparar a auditoria.'
ROOT=Path(os.environ.get('CIV2155_IFC2SB_AUDITORIA',str(Path.cwd())))
ROOT.mkdir(exist_ok=True,parents=True)
REG=ROOT/'registros';COD=ROOT/'codigo';COD.mkdir(exist_ok=True)
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()


IfcOpenShell da auditoria: 0.8.4


In [2]:
HASHES={'compatibilidade_occt.py': 'dc40efd9e9cc2b774242e07d7dc4817e043467e249db86284e393b309299c9c9', 'controle_ifc2sb.py': 'e04ed85b50b9dd73cbdcb466fc794d9721d7c22ae98eefad3b9ec23790d06e96', 'inspecionar_saida.py': 'd5581a8b46dadc784b2fba881c256bc6c72c6f7f9e44e79355f305daedabd370', 'inventariar_entrada.py': '6c6d7968afaf056a7f147472fca34ef75c88553df32c5df9b59652f29b9bfbd7', 'preparar_motor_linux.py': '0481a531b3c53b0e798ad417ecfe0d733459a3841175647113b42210cbd08a7c'}
BASE='https://raw.githubusercontent.com/prcgui/bim-ontology-CIV2155/main/'
for nome,digest in HASHES.items():
    p=COD/nome
    if not p.exists():urlretrieve(BASE+'consultas/ifc2sb_controle_v01/'+nome,p)
    assert sha(p)==digest,nome
pacote=ROOT/'CIV2155_IFC2SB_Controle_Creche_v01_Registros.zip'
if not REG.exists():
    if not pacote.exists():urlretrieve(BASE+'resultados/ifc2sb_controle_v01/'+pacote.name,pacote)
    assert sha(pacote)=='84a38fe8531a16b08c9a8437659dfbd8ca0c6f5798ddb55635e2186d59f7d98b'
    with zipfile.ZipFile(pacote) as f:
        assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in f.namelist())
        f.extractall(REG)
sys.path.insert(0,str(COD))
OUT=ROOT/'auditoria_recalculada';OUT.mkdir(exist_ok=True)
print('Código conferido por SHA-256; registros:',REG)


Código conferido por SHA-256; registros: C:\Users\prcgu\OneDrive\Documentos\ChatGPT\Artigo da Disciplina BIM\desenvolvimento\ifc2sb_retomada_v01\registros


## 2. Entrada FNDE preservada e cobertura de classes

O inventário usa classes e atributos IFC, sem procurar palavras para inventar relações entre ambientes e paredes. A presença de revestimentos fora do filtro do motor constitui uma diferença de cobertura a investigar; não prova a causa de um resultado insatisfatório.


In [3]:
IFC=ROOT/'TIPO2-ARQ-DTV_R03.ifc'
IFC_SHA='0975bf1394694eff8746e5e49232bdf950c36c37c858edb715c9e7bc8f65603a'
if not IFC.exists():
    p=ROOT/'Arquitetura_FNDE_Original.zip'
    if not p.exists():urlretrieve(BASE+'dados/fnde/arquitetura/TIPO2-ARQ-DTV_R03.ifc.zip',p)
    with zipfile.ZipFile(p) as f:IFC.write_bytes(f.read(IFC.name))
assert sha(IFC)==IFC_SHA
from inventariar_entrada import inventariar
inventario=inventariar(IFC,OUT/'inventario_FNDE')


{
  "entrada": "TIPO2-ARQ-DTV_R03.ifc",
  "sha256": "0975bf1394694eff8746e5e49232bdf950c36c37c858edb715c9e7bc8f65603a",
  "schema": "IFC4",
  "filtro_classes_autores": [
    "IfcBeam",
    "IfcColumn",
    "IfcCurtainWall",
    "IfcDoor",
    "IfcMember",
    "IfcPlate",
    "IfcRoof",
    "IfcSlab",
    "IfcWall",
    "IfcWindow"
  ],
  "contagens": {
    "IfcBeam": 0,
    "IfcColumn": 0,
    "IfcCurtainWall": 0,
    "IfcDoor": 53,
    "IfcMember": 0,
    "IfcPlate": 0,
    "IfcRoof": 152,
    "IfcSlab": 0,
    "IfcWall": 457,
    "IfcWindow": 39,
    "IfcSpace": 41,
    "IfcCovering": 631,
    "IfcRelSpaceBoundary": 0,
    "IfcOpeningElement": 184,
    "IfcRelVoidsElement": 184,
    "IfcRelFillsElement": 183
  },
  "covering_tipo_instancia": {
    "CEILING": 291,
    "NOTDEFINED": 191,
    "CLADDING": 149
  },
  "covering_tipo_do_tipo": {
    "CEILING": 291,
    "NOTDEFINED": 191,
    "CLADDING": 149
  },
  "advertencia": "A ausência de IfcCovering no filtro foi verificada no código-

## 3. Resultado do controle: conferir entidades, não apenas o retorno

O controle já possui 7 IfcSpace e 81 limites na entrada. Por isso, apenas contar limites na saída seria insuficiente. Comparam-se classes concretas, GUIDs e espaços relacionados, juntamente com o log de reconstrução. GUID novo sozinho demonstra reescrita, não necessariamente informação nova.


In [4]:
from inspecionar_saida import analisar
C=REG/'01_controle_FZK'
controle=analisar(REG/'dados/AC20-FZK-Haus.ifc',C/'AC20-FZK-Haus_IFC2SB.ifc',OUT/'01_controle_FZK')
mc=json.loads((C/'execucao.json').read_text(encoding='utf8'))
assert mc['entrada_preservada'] and mc['retorno']==0
assert sha(C/'motor.log')==mc['sha256_log']
assert sha(C/'AC20-FZK-Haus_IFC2SB.ifc')==mc['sha256_saida']
assert controle['inventario_saida']['espacos_com_limites']>0
print('Controle operacional com espaços interiores e relações novas; sem validação arquitetônica independente.')


{
  "entrada": "C:\\Users\\prcgu\\OneDrive\\Documentos\\ChatGPT\\Artigo da Disciplina BIM\\desenvolvimento\\ifc2sb_retomada_v01\\registros\\dados\\AC20-FZK-Haus.ifc",
  "sha256_entrada": "ea6f04eaf92fac4d7ad0038bc3d2dfea4c094dd3f516ecc33c50bf1835ca108d",
  "saida": "C:\\Users\\prcgu\\OneDrive\\Documentos\\ChatGPT\\Artigo da Disciplina BIM\\desenvolvimento\\ifc2sb_retomada_v01\\registros\\01_controle_FZK\\AC20-FZK-Haus_IFC2SB.ifc",
  "sha256_saida": "b27245ec3ab5e3aec34300d6c4bf20dde0707623be1f23cce78dee3502ae6827",
  "ifcopenshell_auditoria": "0.8.4",
  "inventario_entrada": {
    "IfcSpace": 7,
    "IfcExternalSpatialElement": 0,
    "IfcWall": 13,
    "IfcSlab": 4,
    "IfcRoof": 0,
    "IfcCovering": 0,
    "IfcDoor": 5,
    "IfcWindow": 11,
    "IfcRelSpaceBoundary": 81,
    "classes_concretas_limites": {
      "IfcRelSpaceBoundary": 81
    },
    "classe_espaco_relacionado": {
      "IfcSpace": 81
    },
    "classe_elemento_relacionado": {
      "IfcWallStandardCase": 28,
      "

Controle operacional com espaços interiores e relações novas; sem validação arquitetônica independente.


O controle produziu **4 espaços interiores**, 131 limites de primeiro nível e 964 de segundo nível. Todos os 1.095 GUIDs de limites diferem dos existentes na entrada; 218 relações referem-se a IfcSpace e 877 ao exterior. Os 7 ambientes originais não foram conservados individualmente: quatro aparecem reunidos em um único espaço. A configuração básica não ativa `--split` nem `--virtualbyspace`.

Portanto, o teste demonstra funcionamento operacional com geração de entidades, mas não preservação da compartimentação desejada. A contagem de limites não é uma taxa de acerto e o log contém avisos que exigem avaliação.


## 4. Creche integral: execução observada e limite de inferência

As opções são as mesmas do controle: `-j2 --graph`. O limite operacional predefinido é 600 segundos. Atingir esse limite não prova impossibilidade do algoritmo nem que o modelo esteja errado. As opções históricas `--first --split --virtualbyspace` não estão ativadas neste ensaio.


In [5]:
F=REG/'02_creche_integral'
mf=json.loads((F/'execucao.json').read_text(encoding='utf8'))
assert mf['sha256_entrada']==IFC_SHA and mf['entrada_preservada']
assert sha(F/'motor.log')==mf['sha256_log']
print(json.dumps(mf,ensure_ascii=False,indent=2))
log=(F/'motor.log').read_text(encoding='utf8',errors='replace')
print('Trecho final do log:\n'+log[-2400:])
saida=F/Path(mf['saida']).name
if mf['arquivo_saida_existe']:
    resultado_fn=analisar(IFC,saida,OUT/'02_creche_integral')
else:
    print('Não há saída IFC auditável neste ensaio. Nenhuma associação ambiente–parede foi promovida ao RDF.')
assert sha(IFC)==IFC_SHA


{
  "caso": "02_creche_integral",
  "artigo": "https://doi.org/10.26868/25222708.2021.30156",
  "codigo_autores": "https://github.com/RWTH-E3D/IFC2SB/tree/96a89cdff65f13279c0c3ce67e729ad9f9f5adad",
  "exe": "/content/civ2155_ifc2sb/IFC2SB_FiltroOriginal",
  "sha256_executavel": "63e8c028956055f93ab973a80f2c8be99c95a2faf96b80fce18979210ba077c9",
  "filtro_classes": "original dos autores; IfcCovering não ativada",
  "entrada": "/content/civ2155_ifc2sb/dados/TIPO2-ARQ-DTV_R03.ifc",
  "sha256_entrada": "0975bf1394694eff8746e5e49232bdf950c36c37c858edb715c9e7bc8f65603a",
  "saida": "/content/civ2155_ifc2sb_retomada_v01/02_creche_integral/TIPO2-ARQ-DTV_R03_IFC2SB.ifc",
  "comando": [
    "/content/civ2155_ifc2sb/IFC2SB_FiltroOriginal",
    "-j2",
    "--graph",
    "/content/civ2155_ifc2sb/dados/TIPO2-ARQ-DTV_R03.ifc",
    "/content/civ2155_ifc2sb_retomada_v01/02_creche_integral/TIPO2-ARQ-DTV_R03_IFC2SB.ifc"
  ],
  "limite_segundos": 600,
  "nota_limite": "Limite operacional predefinido; seu 

## 5. Como executar novamente o programa dos autores

1. Em Linux/Colab, `codigo/preparar_motor_linux.py` compila os commits fixados, aplicando `compatibilidade_occt.py`. A preparação foi extraída do procedimento existente e **não recompilada nesta retomada**: reutilizou-se o binário preservado, cujo SHA-256 consta dos manifestos. Uma nova compilação deve registrar seu próprio ambiente e hash; não se pressupõe identidade binária.
2. `codigo/controle_ifc2sb.py` contém o procedimento efetivamente usado para o controle. Exige o ambiente preparado e o hash do executável desta execução. O caminho do IFC original deve existir e ter o hash declarado.
3. A creche foi iniciada na mesma sessão por `iniciar_caso('02_creche_integral', BASE_MOTOR/'dados/TIPO2-ARQ-DTV_R03.ifc', IFC_SHA, 600)`.
4. Os registros originais estão no [Colab de execução](https://colab.research.google.com/drive/1NDxm9a_XZzLDriUBAKaOQNTChBvwCS0v). Não selecionar "Executar tudo" nesse histórico: há células de compilação e ensaios anteriores.

O Python de supervisão registra comandos, tempo, arquivos e hashes; o cálculo dos limites ocorre no C++ publicado. Uma compilação nova ou outras opções constituem outro ensaio e exigem nova pasta de resultados.


## 6. Consequência para o artigo

O requisito continua sendo representar relações úteis à pergunta de pesquisa: que parte da parede delimita qual ambiente; quais aberturas pertencem a essa parte; quais coordenadas e evidências sustentam cada vínculo. Um volume fechado para simulação não equivale automaticamente ao cômodo arquitetônico desejado.

Os resultados anteriores da adaptação geométrica Python permanecem como experimento auxiliar. Não devem ser apresentados como execução do GRC de Lilis nem como resultado do IFC2SB. A geração completa e validada de relações arquitetônicas para a creche e a integração com a estrutura continuam sem demonstração.
